# Defining a breakout

I will be using this notebook to find a way to accurately define what a breakout is for an NBA player so that I can find out what stats and meatrics I need to track in order to predict future breakouts.

### Threshold crossing

First I will be looking at ppg jumps to help define a breakout season.

In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent)) # project root

from utilities import load_data

df = load_data.load_clean_ppg()
df.shape

(18720, 33)

#### Games-played threshold

Instead of a fixed 40 games, a player needs the same *share* of their season's games (40/82). This keeps shortened seasons (1999, 2012, 2020, 2021) from being treated unfairly. The ppg change is computed on all rows first, then only back-to-back seasons where both years pass the threshold are kept, so skipped years don't create fake jumps.

In [ ]:
MIN_SHARE = 40 / 82   # 40 games in a normal season, about 49% of the season

# Season length: the 99th percentile of games played in that season
# (the max can exceed 82 because traded players' combined rows add up games)
season_len = df.groupby("season")["g"].transform(lambda x: x.quantile(0.99))

# True if the player played enough of THIS season's games
df["enough_games"] = df["g"] >= MIN_SHARE * season_len

In [ ]:
df = df.sort_values(["player_id", "season"])

# Change in ppg vs the previous row for the same player
df["ppg_change"] = df.groupby("player_id")["pts_per_game"].diff()
# How many years apart those two rows are (1 = back-to-back seasons)
df["season_gap"] = df.groupby("player_id")["season"].diff()
# Did the player have enough games in the previous season? (first season -> False)
df["prev_enough"] = df.groupby("player_id")["enough_games"].shift(1, fill_value=False)

# Only compare back-to-back seasons where both seasons have enough games
usable = df[(df["season_gap"] == 1) & df["enough_games"] & df["prev_enough"]]
usable.shape

In [ ]:
# Distribution of ppg jumps: where would a breakout cutoff sit?
display(usable["ppg_change"].describe(percentiles=[.5, .75, .9, .95, .99]))

# The 20 biggest jumps: do these match seasons you'd call breakouts?
usable.sort_values("ppg_change", ascending=False)[["player", "season", "pts_per_game", "ppg_change"]].head(20)

In [ ]:
# Sanity checks
# 1) Thresholds should be lower in shortened seasons (1999, 2012, 2020, 2021)
print((MIN_SHARE * season_len).groupby(df["season"]).first().loc[[1998, 1999, 2011, 2012, 2020, 2021]].round(1))

# 2) Usable rows per season should look similar in neighboring years
print(usable.groupby("season").size().loc[[1998, 1999, 2000, 2011, 2012, 2020, 2021]])

# 3) Share of consecutive-season pairs that are real (season_gap == 1)
print((df["season_gap"] == 1).mean())